<!-- Dashboard metadata hidden for presentation mode -->

In [ ]:
# Uncomment and run the following if you need to install required packages:
# %pip install torch matplotlib numpy ipywidgets jupyter voila

In [ ]:
import torch
import matplotlib.pyplot as plt
import numpy as np
import ipywidgets as widgets
from IPython.display import display, HTML, clear_output

plt.style.use('dark_background')
plt.rcParams['figure.facecolor'] = '#121212'
plt.rcParams['axes.facecolor'] = '#121212'
plt.rcParams['savefig.facecolor'] = '#121212'
plt.rcParams['text.color'] = '#f0f0f0'
plt.rcParams['axes.labelcolor'] = '#f0f0f0'
plt.rcParams['xtick.color'] = '#f0f0f0'
plt.rcParams['ytick.color'] = '#f0f0f0'
plt.rcParams['axes.edgecolor'] = '#444444'

# Try to import demo modules, with fallback mock classes
try:
    from tft_demo import TFTDemo
    from gat_demo import GATDemo
    from fusion_demo import FusionDemo
    from risk_demo import RiskDemo
except ImportError:
    # Fallback mocks for UI demonstration
    class TFTDemo:
        def __call__(self, ts):
            return torch.randn(1, 4)

    class GATDemo:
        def __call__(self, nodes):
            return torch.randn(1, 4)

    class FusionDemo:
        def __call__(self, fused):
            return fused

    class RiskDemo:
        def __call__(self, forecast, price, stock, lead_time, attn):
            return {
                'budget_risk': float(forecast.item() * price.item() / 1000),
                'stock_risk': float(forecast.item() * lead_time.item() / stock.item()),
                'dependency_risk': float(attn.item())
            }

# Initialize models
tft = TFTDemo()
gat = GATDemo()
fusion = FusionDemo()
risk = RiskDemo()

In [ ]:
from IPython.display import HTML

display(HTML('''
<style>
body, .output {
    background-color: #121212 !important;
    color: #f5f5f5 !important;
}
.widget-label, .widget-description, .widget-readout {
    color: #f5f5f5 !important;
}
.jp-Notebook, .jp-NotebookPanel {
    background: #121212 !important;
}
.widget-box, .widget-item {
    background: #1c1c1c !important;
    border: 1px solid #333333 !important;
    color: #f5f5f5 !important;
}
.jupyter-widgets select, .jupyter-widgets option {
    color: #f5f5f5 !important;
    background-color: #1c1c1c !important;
}
.jupyter-widgets input {
    color: #f5f5f5 !important;
    background-color: #1c1c1c !important;
    border: 1px solid #333333 !important;
}
/* Button styles */
.jupyter-widgets button {
    transition: all 0.2s ease;
    color: #f5f5f5 !important;
}
.jupyter-widgets button[data-style="success"]:hover {
    background-color: #4caf50 !important;
    box-shadow: 0 0 10px rgba(76, 175, 80, 0.5) !important;
}
.jupyter-widgets button[data-style="success"]:active {
    background-color: #388e3c !important;
    transform: scale(0.95);
}
.jupyter-widgets button[data-style="danger"]:hover {
    background-color: #f44336 !important;
    box-shadow: 0 0 10px rgba(244, 67, 54, 0.5) !important;
}
.jupyter-widgets button[data-style="danger"]:active {
    background-color: #d32f2f !important;
    transform: scale(0.95);
}
.jupyter-widgets button[data-style="warning"] {
    background-color: #ffeb3b !important;  /* Yellow background */
    color: #000000 !important;  /* Black text for contrast */
}
.jupyter-widgets button[data-style="warning"]:hover {
    background-color: #fdd835 !important;
    box-shadow: 0 0 10px rgba(255, 235, 59, 0.5) !important;
}
.jupyter-widgets button[data-style="warning"]:active {
    background-color: #f9a825 !important;
    transform: scale(0.95);
}
/* Toggle button styling */
.jupyter-widgets .widget-toggle-button button {
    background-color: #1c1c1c !important;  /* Normal background */
    color: #f5f5f5 !important;  /* White text */
    border: 1px solid #333333 !important;
}
.jupyter-widgets .widget-toggle-button button:hover {
    background-color: #2c2c2c !important;
    box-shadow: 0 0 10px rgba(255, 255, 255, 0.1) !important;
}
.jupyter-widgets .widget-toggle-button button:active {
    background-color: #3c3c3c !important;
    transform: scale(0.95);
}
</style>
'''))

# ATSF Dashboard

<div style='background: linear-gradient(135deg, #1e3c72 0%, #2a5298 100%); color: white; padding: 20px; border-radius: 10px; text-align: center; margin-bottom: 20px; box-shadow: 0 4px 6px rgba(0,0,0,0.1);'>
    <h1 style='margin: 0; font-size: 2.5em; font-weight: bold;'>🚀 ATSF Supply Chain Intelligence Dashboard</h1>
    <p style='margin: 10px 0 0 0; font-size: 1.2em; text-align: center;'>Adaptive Temporal-Structural Fusion for Material Demand Forecasting & Risk Assessment</p>
</div>

In [ ]:
# Dummy material list for dropdown
MATERIALS = [
    "Lithium", "Cobalt", "Nickel", "Copper", "Zinc",
    "Aluminium", "Graphite", "Rare Earth Mix"
]

# Predefined scenarios for ATSF
SCENARIOS = [
    "Balanced Data", "Temporal Dominant", "Structural Dominant", "Manual Alpha"
]

<!-- Visualization functions hidden for presentation mode -->

In [ ]:
# Risk Radar Chart
def plot_radar(ax, risks):
    categories = ["Budget", "Stock", "Dependency"]
    values = [risks["budget_risk"], risks["stock_risk"], risks["dependency_risk"]]
    values += values[:1]  # close the loop

    angles = np.linspace(0, 2 * np.pi, len(categories) + 1)

    ax.set_facecolor('#121212')
    ax.set_theta_offset(np.pi / 2)
    ax.set_theta_direction(-1)
    ax.set_thetagrids(angles[:-1] * (180 / np.pi), categories)

    ax.plot(angles, values, linewidth=2, linestyle='solid', color='#80d8ff')
    ax.fill(angles, values, alpha=0.25, color='#80d8ff')
    ax.set_title("Risk Radar Chart", fontsize=14, pad=40, color='#eeeeee')
    ax.tick_params(colors='#bbbbbb')
    plt.tight_layout(pad=3.0)

In [ ]:
# Colored Risk Meter (speedometer)
def get_risk_label(risk_value):
    if risk_value < 0.4:
        return "Low"
    elif risk_value < 0.7:
        return "Medium"
    else:
        return "High"

def plot_risk_meter(ax, risk_score, title="Overall Risk Meter"):
    # clamp 0–1
    r = max(0, min(1, risk_score))

    ax.set_facecolor('#121212')
    ax.axis("off")
    ax.set_title(title, fontsize=14, color='#eeeeee')

    # background arc
    theta = np.linspace(-0.75*np.pi, -0.25*np.pi, 100)
    ax.plot(np.cos(theta), np.sin(theta), color="#444444", linewidth=20)

    # indicator arc (green->yellow->red)
    color = "#66ff66" if r < 0.4 else "#ffd54f" if r < 0.7 else "#ff5252"
    theta_indicator = np.linspace(-0.75*np.pi, -0.75*np.pi + 0.5*np.pi*r, 100)
    ax.plot(np.cos(theta_indicator), np.sin(theta_indicator), color=color, linewidth=20)

    # needle
    angle = -0.75*np.pi + r*(0.5*np.pi)
    ax.plot([0, np.cos(angle)], [0, np.sin(angle)], color="#ffffff", linewidth=3)
    ax.text(0, -0.2, f"{r:.2f}\n({get_risk_label(r)})", fontsize=12, ha="center", color='#ffffff')

In [ ]:
# Side Interpretation Panel
def interpret_risks(risks, current_alpha, scenario, tft_forecast, gat_forecast, fused_forecast, attn_score):
    text = "<h3>📘 Interpretation</h3>"

    text += "<p><b>ATSF Fusion:</b> This model intelligently combines predictions from temporal (TFT) and structural (GAT) patterns, aiming for more robust forecasts.</p>"

    # Scenario-based alpha interpretation
    if scenario == "Temporal Dominant":
        text += f"<p><b>Scenario:</b> '{scenario}' suggests a strong reliance on temporal patterns. The model uses <b>Alpha = {current_alpha:.2f}</b> to prioritize TFT signals.</p>"
    elif scenario == "Structural Dominant":
        text += f"<p><b>Scenario:</b> '{scenario}' suggests a strong reliance on structural relationships. The model uses <b>Alpha = {current_alpha:.2f}</b> to prioritize GAT signals.</p>"
    elif scenario == "Balanced Data":
        text += f"<p><b>Scenario:</b> '{scenario}' indicates a balanced dataset. The model uses <b>Alpha = {current_alpha:.2f}</b> for equal weighting of TFT and GAT.</p>"
    else: # Manual Alpha
        text += f"<p><b>Manual Alpha:</b> You are manually controlling the alpha value. The model uses <b>Alpha = {current_alpha:.2f}</b> for fusion.</p>"

    # Comparison of forecasts
    if fused_forecast > tft_forecast and fused_forecast > gat_forecast:
        text += "<p>📈 <b>ATSF Advantage:</b> The fused forecast is higher than both individual models, suggesting synergy from combining signals.</p>"
    elif fused_forecast < tft_forecast and fused_forecast < gat_forecast:
        text += "<p>📉 <b>ATSF Insight:</b> The fused forecast is lower than both individual models, potentially indicating that fusion has identified a more conservative prediction by reconciling conflicting signals.</p>"
    elif abs(fused_forecast - tft_forecast) < 0.1 and abs(fused_forecast - gat_forecast) < 0.1:
        text += "<p>↔️ <b>ATSF Balance:</b> The fused forecast is close to the individual models, showing a balanced influence without significant divergence.</p>"
    else:
        text += "<p>✨ <b>ATSF Value:</b> Fusion provides a blended forecast, potentially leading to more robust predictions than relying on a single source.</p>"

    # Fusion interpretation (original based on current_alpha)
    if current_alpha > 0.7:
        text += "<p><b>Fusion Weight:</b> Model relies strongly on temporal patterns.</p>"
    elif current_alpha < 0.3:
        text += "<p><b>Fusion Weight:</b> Model relies mostly on structural relationships.</p>"
    else:
        text += "<p><b>Fusion Weight:</b> Balanced use of both signals.</p>"

    # Risk interpretations
    if risks["budget_risk"] > 1:
        text += "<p>💰 <b>Budget Risk:</b> Demand × price exceeds the budget threshold. Action recommended.</p>"
    else:
        text += "<p>💰 <b>Budget Risk:</b> Within safe range.</p>"

    if risks["stock_risk"] > 1:
        text += "<p>📦 <b>Stock Risk:</b> Warning: Lead-time consumption exceeds stock.</p>"
    else:
        text += "<p>📦 <b>Stock Risk:</b> Stock levels are stable.</p>"

    if risks["dependency_risk"] > 0.6:
        if attn_score > 0.8:
            text += f"<p>🔗 <b>Dependency Risk:</b> Critical structural dependency detected. Attention score ({attn_score:.2f}) indicates high vulnerability to supply chain disruptions. Immediate risk mitigation recommended.</p>"
        else:
            text += f"<p>🔗 <b>Dependency Risk:</b> High structural dependency detected. Attention score ({attn_score:.2f}) suggests moderate risk. Monitor closely.</p>"
    else:
        text += f"<p>🔗 <b>Dependency Risk:</b> Dependency stable. Attention score ({attn_score:.2f}) indicates low risk.</p>"

    return HTML(f"""
    <div style='background:#171717; color:#f0f0f0; padding:18px; border-radius:16px; margin-top:10px; border:1px solid #333333; box-shadow:0 0 20px rgba(0,0,0,0.45)'>
        {text}
    </div>
    """)

<!-- Main ATSF logic hidden for presentation mode -->

In [ ]:
def atsf_run(material, forecast_scale, price, stock, lead, alpha_manual, attn_score, scenario, executive_mode):

    clear_output(wait=True)

    # Seed dummy inputs from current controls so plots reflect changes
    seed_input = f"{material}|{scenario}|{forecast_scale:.2f}|{price:.2f}|{stock:.2f}|{lead:.2f}|{attn_score:.2f}"
    seed = abs(hash(seed_input)) % (2**32)
    torch.manual_seed(seed)
    np.random.seed(seed % (2**32))

    # Dummy inputs
    B, T, F = 1, 10, 4
    ts = torch.randn(B, T, F)
    nodes = torch.randn(5, F)
    mat_id = torch.tensor([MATERIALS.index(material)])
    hor_id = torch.tensor([1])

    # Compute embeddings
    h_t = tft(ts)
    h_g = gat(nodes)

    # Individual model forecasts (for comparison)
    

    # Determine the actual alpha for fusion based on scenario
    if scenario == "Temporal Dominant":
        current_alpha = 0.9
    elif scenario == "Structural Dominant":
        current_alpha = 0.1
    elif scenario == "Balanced Data":
        current_alpha = 0.5
    else: # "Manual Alpha"
        current_alpha = alpha_manual

    # Compute individual model forecasts (raw, unweighted)
    tft_forecast = forecast_scale * h_t.norm(dim=1)
    gat_forecast = forecast_scale * h_g.norm(dim=1)

    # Compute fused forecast as alpha-weighted combination of individual forecasts
    forecast = current_alpha * tft_forecast + (1 - current_alpha) * gat_forecast
    
    # Still compute fused embedding for visualization purposes
    alpha_tensor = torch.tensor([[current_alpha]])
    fused = alpha_tensor * h_t + (1 - alpha_tensor) * h_g
    attn_proxy = torch.tensor([attn_score])

    risk_output = risk(
        forecast=forecast,
        price=torch.tensor([price]),
        stock=torch.tensor([stock]),
        lead_time=torch.tensor([lead]),
        attn=attn_proxy
    )

    # --- KPI Cards ---
    kpi_html = f"""
    <div style='display: flex; gap: 20px; margin-bottom: 20px; flex-wrap: wrap;'>
        <div style='background: linear-gradient(135deg, #1e3c72, #2a5298); color: white; padding: 20px; border-radius: 15px; flex: 1; min-width: 200px; text-align: center; box-shadow: 0 4px 8px rgba(0,0,0,0.3);'>
            <h4 style='margin: 0 0 10px 0; font-size: 1.2em;'>📊 Forecast Value</h4>
            <p style='font-size: 2.5em; margin: 0; font-weight: bold;'>{forecast.item():.2f}</p>
            <p style='margin: 5px 0 0 0; font-size: 0.9em; opacity: 0.8;'>ATSF Fused Prediction</p>
        </div>
        <div style='background: linear-gradient(135deg, #ff6b6b, #ee5a52); color: white; padding: 20px; border-radius: 15px; flex: 1; min-width: 200px; text-align: center; box-shadow: 0 4px 8px rgba(0,0,0,0.3);'>
            <h4 style='margin: 0 0 10px 0; font-size: 1.2em;'>⚠️ Overall Risk</h4>
            <p style='font-size: 2.5em; margin: 0; font-weight: bold;'>{max(risk_output.values()):.2f}</p>
            <p style='margin: 5px 0 0 0; font-size: 0.9em; opacity: 0.8;'>Risk Score</p>
        </div>
        <div style='background: linear-gradient(135deg, #4ecdc4, #44a08d); color: white; padding: 20px; border-radius: 15px; flex: 1; min-width: 200px; text-align: center; box-shadow: 0 4px 8px rgba(0,0,0,0.3);'>
            <h4 style='margin: 0 0 10px 0; font-size: 1.2em;'>⚖️ Fusion Alpha</h4>
            <p style='font-size: 2.5em; margin: 0; font-weight: bold;'>{current_alpha:.2f}</p>
            <p style='margin: 5px 0 0 0; font-size: 0.9em; opacity: 0.8;'>TFT vs GAT Weight</p>
        </div>
    </div>
    """
    display(HTML(kpi_html))

    display(executive_toggle)

    if not executive_mode:
        # =====================
        # VISUAL PLOTS
        # =====================
        fig = plt.figure(figsize=(16, 4))
        fig.patch.set_facecolor('#121212')

        # 1. Temporal Embedding
        ax1 = fig.add_subplot(1,3,1)
        ax1.set_facecolor('#121212')
        ax1.stem(h_t.detach().numpy().flatten(), basefmt=" ")
        ax1.set_title(f"Temporal Embedding (TFT) for {material}", color='#f0f0f0')

        # 2. Structural Embedding
        ax2 = fig.add_subplot(1,3,2)
        ax2.set_facecolor('#121212')
        ax2.stem(h_g.detach().numpy().flatten(), basefmt=" ")
        ax2.set_title("Structural Embedding (GAT)", color='#f0f0f0')

        # 3. Fused Embedding
        ax3 = fig.add_subplot(1,3,3)
        ax3.set_facecolor('#121212')
        ax3.stem(fused.detach().numpy().flatten(), basefmt=" ")
        ax3.set_title(f"Fused Embedding (\u03B1 = {current_alpha:.2f})", color='#f0f0f0')

        plt.tight_layout()
        plt.show()

        # --- Forecast Comparison ---
        fig_comp = plt.figure(figsize=(8, 4))
        fig_comp.patch.set_facecolor('#121212')
        ax_comp = fig_comp.add_subplot(1,1,1)
        ax_comp.set_facecolor('#121212')

        x_labels = ['TFT Only', 'GAT Only', 'Fused (ATSF)']
        forecast_values = [tft_forecast.item(), gat_forecast.item(), forecast.item()]
        colors = ['#64b5f6', '#ef9a9a', '#81c784']

        bars = ax_comp.bar(x_labels, forecast_values, color=colors)
        ax_comp.set_title('Forecast Comparison', fontsize=14, color='#f0f0f0')
        ax_comp.set_ylabel('Forecast Value', color='#f0f0f0')
        ax_comp.tick_params(axis='x', rotation=0, colors='#f0f0f0')
        ax_comp.tick_params(axis='y', colors='#f0f0f0')

        for bar in bars:
            yval = bar.get_height()
            ax_comp.text(bar.get_x() + bar.get_width()/2, yval + 0.05, round(yval, 2), ha='center', va='bottom', color='#f0f0f0')

        plt.tight_layout()
        plt.show()

        # Explanatory note on forecast comparison
        forecast_note = f"""
        <div style='background:#1a1a1a; color:#d0d0d0; padding:15px; border-radius:10px; margin-top:15px; border-left:4px solid #80d8ff;'>
            <p style='margin:0; font-size:0.95em; line-height:1.6;'>
                <b>📌 Understanding the Bars:</b> TFT (temporal) and GAT (structural) are <b>independent models</b> that make their own forecasts based on different input patterns. 
                Their individual bar heights reflect how confident each model is, which can vary unpredictably in demo mode. 
                <br/><br/>
                The <b>Fused (ATSF) forecast</b> (green bar) is the <b>alpha-weighted combination</b> of the two individual forecasts, <b>not their sum</b>:
                <ul style='margin:8px 0; padding-left:20px;'>
                    <li><b>Temporal Dominant (α=0.9):</b> 90% TFT + 10% GAT weight</li>
                    <li><b>Balanced Data (α=0.5):</b> 50% TFT + 50% GAT weight</li>
                    <li><b>Structural Dominant (α=0.1):</b> 10% TFT + 90% GAT weight</li>
                    <li><b>Manual Alpha:</b> You control the blend</li>
                </ul>
                The fused forecast = α × (TFT forecast) + (1-α) × (GAT forecast), which creates a balanced prediction that leverages the strengths of both models.
            </p>
        </div>
        """
        display(HTML(forecast_note))

        # --- Individual Risk Meters ---
        fig_meters = plt.figure(figsize=(18, 5))
        fig_meters.patch.set_facecolor('#121212')

        ax_budget = fig_meters.add_subplot(1, 3, 1)
        plot_risk_meter(ax_budget, risk_output['budget_risk'], title="Budget Risk Meter")

        ax_stock = fig_meters.add_subplot(1, 3, 2)
        plot_risk_meter(ax_stock, risk_output['stock_risk'], title="Stock Risk Meter")

        ax_dependency = fig_meters.add_subplot(1, 3, 3)
        plot_risk_meter(ax_dependency, risk_output['dependency_risk'], title="Dependency Risk Meter")

        plt.tight_layout()
        plt.show()

        # --- Radar + Meter ---
        fig2 = plt.figure(figsize=(12,5))
        fig2.patch.set_facecolor('#121212')

        ax4 = fig2.add_subplot(1,2,1, polar=True)
        plot_radar(ax4, risk_output)

        ax5 = fig2.add_subplot(1,2,2)
        ax5.set_facecolor('#121212')
        plot_risk_meter(ax5, max(risk_output.values()), title="Overall Risk Meter")

        plt.tight_layout()
        plt.show()

        # --- Interpretation Panel ---
        display(interpret_risks(risk_output, current_alpha, scenario, tft_forecast.item(), gat_forecast.item(), forecast.item(), attn_score))

    # --- Summary ---
    display(HTML(f"""
    <div style='background:#171717; color:#f0f0f0; padding:18px; border-radius:16px; margin-top:10px; border:1px solid #333333; box-shadow:0 0 20px rgba(0,0,0,0.45)'>
        <h3 style='margin-top:0; color:#80d8ff;'>Summary</h3>
        <div style='display:grid; grid-template-columns:1fr 1fr; gap:12px;'>
            <div style='padding:12px; background:#1f1f1f; border-radius:12px;'>
                <p style='margin:0.2rem 0;'><b>Material:</b> {material}</p>
                <p style='margin:0.2rem 0;'><b>Scenario:</b> {scenario}</p>
                <p style='margin:0.2rem 0;'><b>Fusion Alpha:</b> {current_alpha:.2f}</p>
            </div>
            <div style='padding:12px; background:#1f1f1f; border-radius:12px;'>
                <p style='margin:0.2rem 0;'><b>Forecast (TFT Only):</b> {tft_forecast.item():.2f}</p>
                <p style='margin:0.2rem 0;'><b>Forecast (GAT Only):</b> {gat_forecast.item():.2f}</p>
                <p style='margin:0.2rem 0;'><b>Forecast (Fused ATSF):</b> {forecast.item():.2f}</p>
            </div>
            <div style='padding:12px; background:#1f1f1f; border-radius:12px;'>
                <p style='margin:0.2rem 0;'><b>Budget Risk:</b> {risk_output['budget_risk']:.3f}</p>
                <p style='margin:0.2rem 0;'><b>Stock Risk:</b> {risk_output['stock_risk']:.3f}</p>
                <p style='margin:0.2rem 0;'><b>Dependency Risk:</b> {risk_output['dependency_risk']:.3f}</p>
            </div>
        </div>

    </div>    """))

<!-- Interactive widgets section hidden for presentation mode -->

In [ ]:
# Material and Scenario Dropdowns
material_dd = widgets.Dropdown(
    options=MATERIALS,
    value="Lithium",
    description="Material:",
    style={'description_width':'150px'}
)

scenario_dd = widgets.Dropdown(
    options=SCENARIOS,
    value="Balanced Data",
    description="Scenario:",
    style={'description_width':'150px'}
)

executive_toggle = widgets.ToggleButton(
    value=False,
    description='Executive Summary',
    button_style='info',
    style={'button_color': '#2a5298', 'font_weight': 'bold'},
    layout=widgets.Layout(width='180px')
)


In [ ]:
# Forecast Slider and Text Input
forecast_slider = widgets.FloatSlider(
    value=10, min=1, max=50, step=0.01,
    description="Forecast Scale",
    style={'handle_color': '#4285F4'}
)
forecast_text = widgets.FloatText(
    value=round(forecast_slider.value, 2), min=forecast_slider.min, max=forecast_slider.max, step=0.01,
    description="",
    layout=widgets.Layout(width='100px')
)
widgets.jslink((forecast_slider, 'value'), (forecast_text, 'value'))
forecast_input_group = widgets.HBox([forecast_slider, forecast_text])

# Price Slider and Text Input
price_slider = widgets.FloatSlider(
    value=50, min=1, max=200, step=0.01, description="Price",
    style={'handle_color': '#4285F4'}
)
price_text = widgets.FloatText(
    value=round(price_slider.value, 2), min=price_slider.min, max=price_slider.max, step=0.01,
    description="",
    layout=widgets.Layout(width='100px')
)
widgets.jslink((price_slider, 'value'), (price_text, 'value'))
price_input_group = widgets.HBox([price_slider, price_text])

# Stock Slider and Text Input
stock_slider = widgets.FloatSlider(
    value=200, min=10, max=500, step=0.01, description="Stock",
    style={'handle_color': '#4285F4'}
)
stock_text = widgets.FloatText(
    value=round(stock_slider.value, 2), min=stock_slider.min, max=stock_slider.max, step=0.01,
    description="",
    layout=widgets.Layout(width='100px')
)
widgets.jslink((stock_slider, 'value'), (stock_text, 'value'))
stock_input_group = widgets.HBox([stock_slider, stock_text])

# Lead Time Slider and Text Input
lead_slider = widgets.FloatSlider(
    value=15, min=1, max=60, step=0.01, description="Lead Time",
    style={'handle_color': '#4285F4'}
)
lead_text = widgets.FloatText(
    value=round(lead_slider.value, 2), min=lead_slider.min, max=lead_slider.max, step=0.01,
    description="",
    layout=widgets.Layout(width='100px')
)
widgets.jslink((lead_slider, 'value'), (lead_text, 'value'))
lead_input_group = widgets.HBox([lead_slider, lead_text])

# Alpha Slider and Text Input
alpha_slider = widgets.FloatSlider(
    value=0.5, min=0, max=1, step=0.01, description="Alpha (Fusion)",
    style={'handle_color': '#4285F4'}
)
alpha_text = widgets.FloatText(
    value=round(alpha_slider.value, 2), min=alpha_slider.min, max=alpha_slider.max, step=0.01,
    description="",
    layout=widgets.Layout(width='100px')
)
widgets.jslink((alpha_slider, 'value'), (alpha_text, 'value'))
alpha_input_group = widgets.HBox([alpha_slider, alpha_text])
alpha_input_group.layout.display = 'none'

# Dependency Risk Slider and Text Input
attn_slider = widgets.FloatSlider(
    value=0.3, min=0, max=1, step=0.01, description="Dependency Risk",
    style={'handle_color': '#4285F4'}
)
attn_text = widgets.FloatText(
    value=round(attn_slider.value, 2), min=attn_slider.min, max=attn_slider.max, step=0.01,
    description="",
    layout=widgets.Layout(width='100px')
)
widgets.jslink((attn_slider, 'value'), (attn_text, 'value'))
attn_input_group = widgets.HBox([attn_slider, attn_text])

# Round any manually entered float values to 2 decimals
def round_to_two_decimals(change):
    if change['name'] == 'value' and change['new'] is not None:
        owner = change['owner']
        owner.value = round(owner.value, 2)

for widget in [forecast_text, price_text, stock_text, lead_text, alpha_text, attn_text]:
    widget.observe(round_to_two_decimals, names='value')

In [ ]:
# Reset Button and Logic
reset_button = widgets.Button(
    description="Reset",
    button_style='warning',
    layout=widgets.Layout(width='auto', flex='1 1 auto')
)

def reset_sliders(b):
    material_dd.value = "Lithium"
    scenario_dd.value = "Balanced Data"
    forecast_slider.value = 10
    price_slider.value = 50
    stock_slider.value = 200
    lead_slider.value = 15
    alpha_slider.value = 0.5
    attn_slider.value = 0.3

reset_button.on_click(reset_sliders)

# Scenario Buttons
safe_button = widgets.Button(
    description="Safe Scenario",
    button_style='success',
    layout=widgets.Layout(width='auto')
)

def set_safe_scenario(b):
    forecast_slider.value = 5
    price_slider.value = 10
    stock_slider.value = 500
    lead_slider.value = 5
    attn_slider.value = 0.2
    scenario_dd.value = "Balanced Data"

safe_button.on_click(set_safe_scenario)

high_budget_button = widgets.Button(
    description="High Budget Risk",
    button_style='danger',
    layout=widgets.Layout(width='auto')
)

def set_high_budget_risk(b):
    forecast_slider.value = 40
    price_slider.value = 150
    stock_slider.value = 200
    lead_slider.value = 15
    attn_slider.value = 0.3
    scenario_dd.value = "Balanced Data"

high_budget_button.on_click(set_high_budget_risk)

high_stock_button = widgets.Button(
    description="High Stock Risk",
    button_style='danger',
    layout=widgets.Layout(width='auto')
)

def set_high_stock_risk(b):
    forecast_slider.value = 40
    price_slider.value = 50
    stock_slider.value = 50
    lead_slider.value = 50
    attn_slider.value = 0.3
    scenario_dd.value = "Balanced Data"

high_stock_button.on_click(set_high_stock_risk)

high_dependency_button = widgets.Button(
    description="High Dependency Risk",
    button_style='danger',
    layout=widgets.Layout(width='auto')
)

def set_high_dependency_risk(b):
    forecast_slider.value = 10
    price_slider.value = 50
    stock_slider.value = 200
    lead_slider.value = 15
    attn_slider.value = 0.8
    scenario_dd.value = "Balanced Data"

high_dependency_button.on_click(set_high_dependency_risk)

high_overall_button = widgets.Button(
    description="High Overall Risk",
    button_style='danger',
    layout=widgets.Layout(width='auto')
)

def set_high_overall_risk(b):
    forecast_slider.value = 40
    price_slider.value = 150
    stock_slider.value = 50
    lead_slider.value = 50
    attn_slider.value = 0.8
    scenario_dd.value = "Balanced Data"

high_overall_button.on_click(set_high_overall_risk)

In [ ]:
# Build UI Layout
scenario_buttons = widgets.HBox([
    safe_button,
    high_budget_button,
    high_stock_button,
    high_dependency_button,
    high_overall_button
], layout=widgets.Layout(flex_wrap='wrap'))

ui = widgets.VBox([
    material_dd,
    scenario_dd,
    forecast_input_group,
    price_input_group,
    stock_input_group,
    lead_input_group,
    alpha_input_group,
    attn_input_group,
    scenario_buttons,
    reset_button
])

# Create interactive output
out = widgets.interactive_output(
    atsf_run,
    {
        "material": material_dd,
        "forecast_scale": forecast_slider,
        "price": price_slider,
        "stock": stock_slider,
        "lead": lead_slider,
        "alpha_manual": alpha_slider,
        "attn_score": attn_slider,
        "scenario": scenario_dd,
        "executive_mode": executive_toggle
    }
)

# Update alpha based on scenario
def update_alpha_from_scenario(change):
    if change['new'] == "Temporal Dominant":
        alpha_slider.value = 0.9
    elif change['new'] == "Structural Dominant":
        alpha_slider.value = 0.1
        alpha_input_group.layout.display = 'none'
    elif change['new'] == "Balanced Data":
        alpha_slider.value = 0.5
        alpha_input_group.layout.display = 'none'
    elif change['new'] == "Manual Alpha":
        alpha_input_group.layout.display = 'flex'
    else:
        alpha_input_group.layout.display = 'none'

scenario_dd.observe(update_alpha_from_scenario, names='value')

# Display Dashboard
panel = widgets.VBox([
    widgets.HTML("<h2 style='color:#80d8ff; margin-bottom:10px;'>ATSF Dashboard</h2>"),
    ui,
    out
])

display(panel)

<!-- Launch Dashboard in Browser instructions hidden for presentation mode -->

In [ ]:
import subprocess
import sys
import time
import webbrowser
from pathlib import Path

notebook_path = Path("ATSF_Dashboard.ipynb")

In [ ]:
launch_button = widgets.Button(
    description="🚀 Launch Dashboard in Browser",
    button_style='success',
    layout=widgets.Layout(width='280px', height='40px')
)
launch_output = widgets.Output()

def launch_voila(b):
    with launch_output:
        launch_output.clear_output()
        try:
            subprocess.run([sys.executable, '-m', 'voila', '--version'], capture_output=True, check=True, text=True)
        except Exception:
            print('Voila is not installed. Run: pip install voila')
            return

        print('Starting Voila...')
        try:
            subprocess.Popen(
                [sys.executable, '-m', 'voila', str(notebook_path), '--no-browser'],
                stdout=subprocess.PIPE,
                stderr=subprocess.PIPE
            )
            time.sleep(2)
            webbrowser.open('http://localhost:8866')
            print('Browser should open at http://localhost:8866')
        except Exception as e:
            print('Failed to launch Voila:', e)

launch_button.on_click(launch_voila)

display(widgets.VBox([launch_button, launch_output]))
